# Fine-Tune DistilBERT on RavenPack Headlines (TRNA Substitute)

**Goal:** adapt the PhraseBank-trained DistilBERT classifier to RavenPack news headlines, using RavenPack `event_sentiment_score` as the supervision signal (our TRNA substitute).

Companion module: `src/sentiment_ltr/models/ravenpack_sentiment.py`  
Plan reference: `docs/news_sentiment_finetuning_plan.md` (Iteration 4)

---

## Inputs

| Input | Location / detail |
| --- | --- |
| **RavenPack article export** | `data/raw/news/ravenpack/{ticker}_articles_2003_2014.parquet` — built by `notebooks/fetch_news_articles.ipynb` |
| **Required columns** | `headline`, `event_sentiment_score`, `article_date` (or `article_time`) |
| **Starting checkpoint** (recommended) | `data/models/phrasebank_distilbert_best/` — DistilBERT fine-tuned on Financial PhraseBank |
| **Fallback init** | `distilbert-base-uncased` if no PhraseBank checkpoint exists |
| **Label mapping** | `event_sentiment_score` → `negative` / `neutral` / `positive` (threshold ±0.05) |
| **Train / val / test split** | Time-based: train ≤ 2011, validation = 2012, test ≥ 2013 |
| **Hyperparameters** | `lr=2e-5`, `batch_size=16`, `max_length=128`, `epochs=2` (default) |

**Prerequisites:** conda env `sentiment-ltr-paper` with `requirements-finetuning.txt` installed; at least one ticker with a rich RavenPack export (currently **AAPL**: ~69k labeled headlines).

---

## Steps

1. **Environment check** — confirm `torch`, `transformers`, `datasets`, and GPU/MPS device.
2. **Discover data** — list `*_articles_*.parquet` under `data/raw/news/ravenpack/`; pick ticker(s).
3. **Load & inspect** — labeled rows (headline + score), class balance, split row counts.
4. **Build HF dataset** — map headlines to `sentence`, scores to 3-class `label`; time-based splits.
5. **Load init checkpoint** — PhraseBank weights (recommended) or `distilbert-base-uncased`.
6. **Tokenize** — `max_length=128`, fixed padding (same as PhraseBank pipeline).
7. **Train** — Hugging Face `Trainer`, 2 epochs, eval each epoch, `load_best_model_at_end` on **validation macro-F1**.
8. **Evaluate** — report validation + **test** accuracy and macro-F1.
9. **Save** — write checkpoint + `metrics.json` for the app / batch scoring.

---

## Expected outputs

| Output | Path / description |
| --- | --- |
| **Fine-tuned model** | `data/models/ravenpack_distilbert_best/` (`config.json`, tokenizer, weights) |
| **Training metrics** | `data/models/ravenpack_distilbert_best/metrics.json` — train loss, val/test `eval_f1`, `eval_accuracy`, hyperparams, tickers used |
| **Console summary** | Split sizes (e.g. AAPL: ~38k train / ~13k val / ~18k test), class balance, final test macro-F1 |
| **Downstream use** | Sentiment Lab → *Fine-tune on RavenPack headlines*; future batch-scoring of cached headlines (Iteration 4.2) |

**Success criteria:** training completes without error; test macro-F1 is reported; checkpoint loads for inference on new headlines.


In [ ]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

%load_ext autoreload
%autoreload 2

from sentiment_ltr.models.phrasebank_sentiment import (
    DEFAULT_MODEL_DIR,
    MAX_LENGTH,
    MODEL_NAME,
    label_maps,
    load_classifier,
    load_metrics,
    load_phrasebank,
    model_is_saved,
    predict_sentences,
)

N = 2  # samples per split

# 1. Load PhraseBank-trained checkpoint
if not model_is_saved(DEFAULT_MODEL_DIR):
    raise FileNotFoundError(
        f"No checkpoint at {DEFAULT_MODEL_DIR}. Run notebooks/liquidAI_prep.ipynb first."
    )
tokenizer, model, device = load_classifier(DEFAULT_MODEL_DIR)

# How was this model tokenized?
# - Tokenizer object: loaded from the checkpoint dir (tokenizer.json + vocab saved at train time).
# - Model config: architecture + label head only — it does NOT store padding/truncation rules.
# - Training contract: metrics.json + phrasebank_sentiment.tokenize_dataset().
metrics = load_metrics(DEFAULT_MODEL_DIR)
train_max_length = metrics.get("max_length", MAX_LENGTH)
base_model = metrics.get("model_name", MODEL_NAME)

pd.DataFrame(
    {
        "setting": [
            "base_model (HF hub id)",
            "tokenizer_class",
            "tokenizer_loaded_from",
            "model_type",
            "vocab_size",
            "padding_side",
            "tokenizer.model_max_length",
            "train/infer max_length",
            "training truncation",
            "training padding",
            "inference padding (predict_sentences)",
        ],
        "value": [
            base_model,
            type(tokenizer).__name__,
            tokenizer.name_or_path,
            model.config.model_type,
            model.config.vocab_size,
            tokenizer.padding_side,
            tokenizer.model_max_length,
            train_max_length,
            True,
            "max_length",
            "longest in batch (padding=True)",
        ],
    }
)

# 2. Sample inferences on PhraseBank validation & test (splits the model was trained on)
raw = load_phrasebank()
_, id2label, _ = label_maps(raw)

val = raw["validation"].shuffle(seed=42).select(range(N))
test = raw["test"].shuffle(seed=42).select(range(N))

samples = pd.concat(
    [
        pd.DataFrame({
            "split": "validation",
            "sentence": val["sentence"],
            "label": [id2label[int(i)] for i in val["label"]],
        }),
        pd.DataFrame({
            "split": "test",
            "sentence": test["sentence"],
            "label": [id2label[int(i)] for i in test["label"]],
        }),
    ],
    ignore_index=True,
)

preds = predict_sentences(samples["sentence"].tolist(), tokenizer, model, device)

# Training-style tokenization for the first sample (fixed length 128)
example = samples["sentence"].iloc[0]
enc = tokenizer(
    example,
    truncation=True,
    padding="max_length",
    max_length=train_max_length,
)
pd.DataFrame(
    {
        "token": tokenizer.convert_ids_to_tokens(enc["input_ids"]),
        "input_id": enc["input_ids"],
        "attention_mask": enc["attention_mask"],
    }
)

samples.join(preds.drop(columns=["sentence"])).assign(
    match=lambda df: df["label"] == df["pred"]
)


In [ ]:
from sentiment_ltr.models.phrasebank_sentiment import phrasebank_probability_chart_frame
from sentiment_ltr.viz import split_series_distribution_figures

CLASS_ORDER = ["negative", "neutral", "positive"]

long_probs = phrasebank_probability_chart_frame()
split_order = long_probs["split"].cat.categories.tolist()
chart_orders = {"split": split_order, "class": CLASS_ORDER}
chart_labels = {
    "split": "PhraseBank split",
    "probability": "Predicted probability",
    "p50": "Median predicted probability",
    "class": "Class",
}

fig_box, fig_p50, p50 = split_series_distribution_figures(
    long_probs,
    x="split",
    y="probability",
    series="class",
    category_orders=chart_orders,
    axis_labels=chart_labels,
    box_title="Class probabilities by split (box & whisker)",
    median_title="Median class probability by split (p50)",
    median_col="p50",
    x_hover_label="PhraseBank split",
    series_hover_label="Class",
    y_hover_label="Predicted probability",
    median_y_hover_label="Median predicted probability",
)
fig_box.show()
fig_p50.show()


### How was original model tokenized

### Schema of the labels the original model was trained on

In [ ]:
from transformers import AutoConfig

from sentiment_ltr.models.ravenpack_sentiment import ID2LABEL as RP_ID2LABEL

# Label schema baked into the PhraseBank checkpoint (starting weights for RavenPack)
if not model_is_saved(DEFAULT_MODEL_DIR):
    raise FileNotFoundError(
        f"No checkpoint at {DEFAULT_MODEL_DIR}. Run notebooks/liquidAI_prep.ipynb first."
    )

pb_config = AutoConfig.from_pretrained(DEFAULT_MODEL_DIR)
checkpoint_id2label = {int(k): v for k, v in pb_config.id2label.items()}   # {0: "negative", ...}
checkpoint_label2id = {k: int(v) for k, v in pb_config.label2id.items()}   # {"negative": 0, ...}

raw = load_phrasebank()
_, dataset_id2label, dataset_label2id = label_maps(raw)

print("PhraseBank checkpoint config (model.config)")
display(
    pd.DataFrame(
        {"id": list(checkpoint_id2label.keys()), "label": list(checkpoint_id2label.values())}
    ).set_index("id")
)
print("label2id:", checkpoint_label2id)

print("\nPhraseBank HF dataset ClassLabel (should match checkpoint)")
display(
    pd.DataFrame(
        {"id": list(dataset_id2label.keys()), "label": list(dataset_id2label.values())}
    ).set_index("id")
)

print("\nRavenPack fine-tune target labels (same 3-class schema)")
display(
    pd.DataFrame(
        {"id": list(RP_ID2LABEL.keys()), "label": list(RP_ID2LABEL.values())}
    ).set_index("id")
)

assert checkpoint_id2label == dataset_id2label == RP_ID2LABEL
print("\n✓ All three id2label maps match — RavenPack labels align with PhraseBank head.")

## RavenPack inputs & outputs (AAPL)

Loads the rich local export from `notebooks/fetch_news_articles.ipynb`:

| Stage | What | Detail |
| --- | --- | --- |
| **Input file** | `data/raw/news/ravenpack/aapl_articles_2003_2014.parquet` | ~311k article rows × 23 columns (headline, scores, prices, metadata) |
| **Required columns** | `headline`, `event_sentiment_score`, `article_date` | Used by `load_ravenpack_labeled_frame()` |
| **Label rule** | `event_sentiment_score` → 3 classes | `> +0.05` positive, `< -0.05` negative, else neutral |
| **Labeled frame** | Deduped rows with `label_name` + `label` | ~69k usable headlines for fine-tuning |
| **HF output** | `DatasetDict` with `sentence` + `label` | Time split: train ≤2011, val 2012, test ≥2013 |

Module: `src/sentiment_ltr/models/ravenpack_sentiment.py`


In [ ]:
from IPython.display import display

from sentiment_ltr.models.ravenpack_sentiment import (
    RAVENPACK_NEWS_DIR,
    SENTIMENT_SCORE_THRESHOLD,
    SPLIT_SOURCE,
    discover_ravenpack_article_files,
    load_ravenpack_labeled_frame,
    ravenpack_class_balance,
    ravenpack_split_summary,
    ravenpack_to_hf_dataset,
)

TICKER = "AAPL"

# ── Input: local parquet export ─────────────────────────────────────────────
article_paths = discover_ravenpack_article_files([TICKER])
if not article_paths:
    raise FileNotFoundError(
        f"No RavenPack export for {TICKER} under {RAVENPACK_NEWS_DIR}. "
        "Run notebooks/fetch_news_articles.ipynb first."
    )
article_path = article_paths[0]

print("INPUT")
print(f"  file:        {article_path.relative_to(PROJECT_ROOT)}")
print(f"  label rule:  |event_sentiment_score| > {SENTIMENT_SCORE_THRESHOLD} → pos/neg; else neutral")
print(f"  splits:      {SPLIT_SOURCE}\n")

raw_rp = pd.read_parquet(article_path)
display(
    pd.Series(
        {
            "rows": f"{len(raw_rp):,}",
            "columns": raw_rp.shape[1],
            "date_min": str(raw_rp["article_date"].min()),
            "date_max": str(raw_rp["article_date"].max()),
        },
        name="raw export summary",
    ).to_frame("value")
)
print("Raw column dtypes:")
display(raw_rp.dtypes.to_frame("dtype"))
print("Sample raw rows (training-relevant columns):")
display(
    raw_rp[
        ["ticker", "article_date", "headline", "event_sentiment_score", "relevance_score", "story_id"]
    ].head(5)
)

# ── Labeled frame (module output) ─────────────────────────────────────────
labeled = load_ravenpack_labeled_frame([TICKER])

print("\nLABELED FRAME (after score→label, headline filter, dedupe)")
display(
    pd.Series(
        {
            "rows": f"{len(labeled):,}",
            "dropped_from_raw": f"{len(raw_rp) - len(labeled):,}",
            "columns": labeled.shape[1],
        },
        name="labeled summary",
    ).to_frame("value")
)
display(ravenpack_class_balance(labeled))
display(ravenpack_split_summary(labeled))
print("Sample labeled rows:")
display(
    labeled[
        ["ticker", "article_date", "headline", "event_sentiment_score", "label_name", "label"]
    ].head(5)
)

# ── Hugging Face DatasetDict (fine-tuning input) ────────────────────────────
hf_ds = ravenpack_to_hf_dataset(labeled)

print("\nHF DATASET OUTPUT (what train_ravenpack() consumes)")
hf_summary = pd.DataFrame(
    {
        "split": list(hf_ds.keys()),
        "rows": [len(hf_ds[s]) for s in hf_ds],
        "columns": [", ".join(hf_ds[s].column_names) for s in hf_ds],
    }
)
display(hf_summary)
print("Sample train rows (`sentence` = headline, `label` = 0/1/2):")
display(hf_ds["train"].to_pandas().head(5))


## AAPL inference: predicted vs RavenPack actual labels

Score headlines with the best available checkpoint (RavenPack if trained, else PhraseBank) and compare `pred` to RavenPack `label_name` (`event_sentiment_score` → negative / neutral / positive).

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

from sentiment_ltr.models.ravenpack_sentiment import (
    DEFAULT_RAVENPACK_MODEL_DIR,
    ID2LABEL,
    assign_time_split,
    ravenpack_model_is_saved,
)

# ── Config ──────────────────────────────────────────────────────────────────
# Out-of-box = PhraseBank checkpoint applied to RavenPack headlines (no RavenPack fine-tune)
FORCE_CHECKPOINT = "phrasebank"  # "phrasebank" | "ravenpack" | "auto"
EVAL_SPLIT = "test"  # train | validation | test | None
BATCH_SIZE = 64
MAX_ROWS = None  # e.g. 2000 for a quick smoke test

if FORCE_CHECKPOINT == "phrasebank":
    model_dir = DEFAULT_MODEL_DIR
    ckpt_label = "PhraseBank (out-of-box on RavenPack)"
elif FORCE_CHECKPOINT == "ravenpack":
    if not ravenpack_model_is_saved():
        raise FileNotFoundError("No RavenPack checkpoint — train first or use phrasebank")
    model_dir = DEFAULT_RAVENPACK_MODEL_DIR
    ckpt_label = "RavenPack fine-tuned"
else:
    use_ravenpack_ckpt = ravenpack_model_is_saved()
    model_dir = DEFAULT_RAVENPACK_MODEL_DIR if use_ravenpack_ckpt else DEFAULT_MODEL_DIR
    ckpt_label = "RavenPack fine-tuned" if use_ravenpack_ckpt else "PhraseBank (out-of-box)"

print(f"Checkpoint: {ckpt_label}")
print(f"  path: {model_dir.relative_to(PROJECT_ROOT)}")

tokenizer, model, device = load_classifier(model_dir)

eval_df = labeled.copy()
eval_df["split"] = assign_time_split(eval_df["article_date"])
if EVAL_SPLIT:
    eval_df = eval_df[eval_df["split"] == EVAL_SPLIT].reset_index(drop=True)
if MAX_ROWS and len(eval_df) > MAX_ROWS:
    eval_df = eval_df.sample(MAX_ROWS, random_state=42).reset_index(drop=True)

print(f"\nScoring {len(eval_df):,} {TICKER} headlines" + (f" ({EVAL_SPLIT} split)" if EVAL_SPLIT else ""))

headlines = eval_df["headline"].tolist()
pred_chunks: list[pd.DataFrame] = []
for start in range(0, len(headlines), BATCH_SIZE):
    batch = headlines[start : start + BATCH_SIZE]
    pred_chunks.append(predict_sentences(batch, tokenizer, model, device))
preds = pd.concat(pred_chunks, ignore_index=True)

results = eval_df[
    ["split", "article_date", "headline", "event_sentiment_score", "label_name"]
].join(preds.drop(columns=["sentence"]))
results = results.rename(columns={"label_name": "actual"})
results["match"] = results["actual"] == results["pred"]

label_order = [ID2LABEL[i] for i in sorted(ID2LABEL)]
y_true = results["actual"]
y_pred = results["pred"]

acc = accuracy_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, labels=label_order, average="macro", zero_division=0)

print("\n=== Performance metrics ===")
print(f"  accuracy:  {acc:.1%}")
print(f"  macro-F1:  {macro_f1:.1%}")
print("\nClassification report (actual = RavenPack score labels):")
print(classification_report(y_true, y_pred, labels=label_order, digits=3, zero_division=0))

cm = pd.DataFrame(
    confusion_matrix(y_true, y_pred, labels=label_order),
    index=pd.Index(label_order, name="actual"),
    columns=pd.Index(label_order, name="pred"),
)
cm_pct = cm.div(cm.sum(axis=1), axis=0).mul(100)

display(
    cm.style.format("{:,.0f}")
    .bar(align="mid", color=["red", "lightgreen"])
    .set_caption(f"Confusion matrix — counts ({ckpt_label}, {TICKER} {EVAL_SPLIT})")
)
display(
    cm_pct.round(1)
    .style.format("{:.1f}%")
    .bar(align="mid", color=["red", "lightgreen"], vmin=0, vmax=100)
    .set_caption("Confusion matrix — % of actual class (row-normalized)")
)

print("Random mismatches:")
mismatches = results.loc[~results["match"]]
display(
    mismatches.sample(min(10, len(mismatches)), random_state=42)[
        [
            "article_date",
            "headline",
            "event_sentiment_score",
            "actual",
            "pred",
            "p(negative)",
            "p(neutral)",
            "p(positive)",
        ]
    ]
)

## Label distribution: PhraseBank (train) vs RavenPack (inference)

Compares class prevalence across three views:
1. **PhraseBank** — the domain the model was trained on (per split + total)
2. **RavenPack actual** — true labels derived from `event_sentiment_score` thresholding (out-of-domain)
3. **RavenPack predicted** — what the model actually outputs on the eval split

Large gaps between PhraseBank and RavenPack actual reveal label-distribution shift.
Large gaps between actual and predicted reveal model bias toward the training distribution.

In [ ]:
import plotly.graph_objects as go

_CLASS_ORDER = ["negative", "neutral", "positive"]

# ── 1. PhraseBank distribution per split (training domain) ───────────────────
pb_rows = []
for split_name in ["train", "validation", "test"]:
    ds_split = raw[split_name].to_pandas()
    total    = len(ds_split)
    counts   = ds_split["label"].map(dataset_id2label).value_counts()
    for cls in _CLASS_ORDER:
        n = int(counts.get(cls, 0))
        pb_rows.append({"split": split_name, "label": cls, "count": n, "pct": 100 * n / total})
pb_dist = pd.DataFrame(pb_rows)

pb_total = (
    pb_dist.groupby("label", as_index=False)[["count"]].sum()
    .assign(pct=lambda d: d["count"] / d["count"].sum() * 100, split="ALL")
)

# ── 2. RavenPack actual distribution (all splits — full labeled frame) ────────
rp_actual_vc = labeled["label_name"].value_counts().reindex(_CLASS_ORDER, fill_value=0)
rp_actual = pd.DataFrame({
    "label": _CLASS_ORDER,
    "count": rp_actual_vc.values.tolist(),
}).assign(pct=lambda d: d["count"] / d["count"].sum() * 100)

# ── 3. RavenPack predicted distribution (eval split from previous cell) ───────
rp_pred_vc = results["pred"].value_counts().reindex(_CLASS_ORDER, fill_value=0)
rp_pred = pd.DataFrame({
    "label": _CLASS_ORDER,
    "count": rp_pred_vc.values.tolist(),
}).assign(pct=lambda d: d["count"] / d["count"].sum() * 100)

# ── Summary table ─────────────────────────────────────────────────────────────
print("=== PhraseBank label distribution by split ===")
display(
    pb_dist.pivot_table(index="label", columns="split", values=["count", "pct"], aggfunc="sum")
    .reindex(_CLASS_ORDER)
    .round(1)
)

cmp = pd.DataFrame({
    "PhraseBank total":              pb_total.set_index("label")["count"].reindex(_CLASS_ORDER),
    "PhraseBank total %":            pb_total.set_index("label")["pct"].reindex(_CLASS_ORDER).round(1),
    "RavenPack actual count":        rp_actual.set_index("label")["count"].reindex(_CLASS_ORDER),
    "RavenPack actual %":            rp_actual.set_index("label")["pct"].reindex(_CLASS_ORDER).round(1),
    f"RavenPack predicted count":    rp_pred.set_index("label")["count"].reindex(_CLASS_ORDER),
    f"RavenPack predicted %":        rp_pred.set_index("label")["pct"].reindex(_CLASS_ORDER).round(1),
})
print(f"\n=== Totals comparison (RavenPack eval split: {EVAL_SPLIT}, checkpoint: {ckpt_label}) ===")
display(cmp)

# ── Bar chart: three-way comparison ──────────────────────────────────────────
fig = go.Figure()
traces = [
    ("PhraseBank (all splits)", pb_total.set_index("label").reindex(_CLASS_ORDER)),
    ("RavenPack actual (all splits)", rp_actual.set_index("label").reindex(_CLASS_ORDER)),
    (f"RavenPack predicted ({ckpt_label}, {EVAL_SPLIT})", rp_pred.set_index("label").reindex(_CLASS_ORDER)),
]
for name, df in traces:
    fig.add_trace(go.Bar(
        name=name,
        x=_CLASS_ORDER,
        y=df["pct"].tolist(),
        text=[f"{p:.1f}%<br>n={n:,}" for p, n in zip(df["pct"], df["count"])],
        textposition="outside",
    ))

fig.update_layout(
    barmode="group",
    title="Label distribution shift: PhraseBank (train) → RavenPack (out-of-domain)",
    xaxis_title="Sentiment class",
    yaxis=dict(title="% of dataset", range=[0, 105]),
    legend=dict(
        orientation="v",
        yanchor="middle", y=0.5,
        xanchor="left",   x=1.02,
    ),
    height=500,
    margin=dict(t=60, r=280),
)
fig.show()

# ── Bar chart: PhraseBank per-split breakdown ─────────────────────────────────
fig2 = go.Figure()
for split_name in ["train", "validation", "test"]:
    sdf = pb_dist[pb_dist["split"] == split_name].set_index("label").reindex(_CLASS_ORDER)
    fig2.add_trace(go.Bar(
        name=f"PhraseBank {split_name}",
        x=_CLASS_ORDER,
        y=sdf["pct"].tolist(),
        text=[f"{p:.1f}%<br>n={n:,}" for p, n in zip(sdf["pct"], sdf["count"])],
        textposition="outside",
    ))
fig2.update_layout(
    barmode="group",
    title="PhraseBank label distribution by split",
    xaxis_title="Sentiment class",
    yaxis=dict(title="% of split", range=[0, 105]),
    height=430,
)
fig2.show()

## Model provenance & reproducibility snapshot

To be able to reproduce or audit the exact checkpoint evaluated above
(`ckpt_label`, `model_dir`), capture:

1. **Revision hash** — git commit of this codebase at run time (+ dirty flag).
2. **Weights snapshot** — sha256 checksum of each saved weight file (detects any
   accidental re-save, corruption, or silent overwrite).
3. **Config info** — `model.config.to_dict()`, `num_labels`, `id2label`.
4. **Tokenizer settings** — `max_length` and padding strategy used at train time.
5. **Data provenance** — dataset, split sizes, training seed, and a content
   hash per split (detects silent upstream changes to the training data).

Branches on whether `model_dir` is the PhraseBank checkpoint or the RavenPack
fine-tuned checkpoint, since their training data differs. Everything is written
to `provenance.json` next to `metrics.json` in `model_dir`.

In [ ]:
import subprocess


def _git_info(repo_path: Path = PROJECT_ROOT) -> dict:
    """Git commit hash + dirty-tree flag for the code that produced this run."""

    def _run(args: list[str]) -> str | None:
        try:
            return subprocess.check_output(args, cwd=repo_path, text=True).strip()
        except Exception:
            return None

    dirty_files = _run(["git", "status", "--porcelain"])
    return {
        "commit_hash": _run(["git", "rev-parse", "HEAD"]),
        "commit_hash_short": _run(["git", "rev-parse", "--short", "HEAD"]),
        "branch": _run(["git", "rev-parse", "--abbrev-ref", "HEAD"]),
        "is_dirty": bool(dirty_files),
        "dirty_files": dirty_files.splitlines() if dirty_files else [],
    }


git_info = _git_info()
git_info

In [ ]:
import hashlib


def _hash_file(path: Path, chunk_size: int = 1 << 20) -> str:
    """sha256 of a file's bytes — cheap integrity check for saved weights."""
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()


weight_files = sorted(
    p for p in model_dir.glob("*") if p.suffix in {".safetensors", ".bin"}
)
weights_snapshot = [
    {"file": p.name, "size_bytes": p.stat().st_size, "sha256": _hash_file(p)}
    for p in weight_files
]
pd.DataFrame(weights_snapshot)

In [ ]:
config_dict = model.config.to_dict()
config_info = {
    "num_labels": model.config.num_labels,
    "id2label": model.config.id2label,
    "label2id": model.config.label2id,
    "architectures": config_dict.get("architectures"),
    "model_type": config_dict.get("model_type"),
    "full_config": config_dict,
}
print("num_labels:", config_info["num_labels"])
print("id2label  :", config_info["id2label"])
print("label2id  :", config_info["label2id"])

In [ ]:
ckpt_metrics = load_metrics(model_dir)
tokenizer_settings = {
    "tokenizer_class": type(tokenizer).__name__,
    "max_length_used": ckpt_metrics.get("max_length", MAX_LENGTH),
    "padding_strategy": "max_length",  # training-time padding (tokenize_dataset helpers)
    "truncation": True,
    "model_max_length": tokenizer.model_max_length,
    "vocab_size": tokenizer.vocab_size,
}
tokenizer_settings

In [ ]:
def _hash_pairs(texts, labels) -> str:
    """Stable content hash over (text, label) pairs — detects silent upstream
    changes to the training data between runs."""
    h = hashlib.sha256()
    for t, l in zip(texts, labels):
        h.update(str(t).encode("utf-8"))
        h.update(b"\x00")
        h.update(str(l).encode("utf-8"))
        h.update(b"\x01")
    return h.hexdigest()


if model_dir == DEFAULT_RAVENPACK_MODEL_DIR:
    # RavenPack fine-tuned checkpoint — trained on the RavenPack labeled frame.
    rp_splits = labeled.copy()
    rp_splits["split"] = assign_time_split(rp_splits["article_date"])
    data_provenance = {
        "dataset_repo": f"RavenPack via WRDS ({TICKER})",
        "dataset_config": SPLIT_SOURCE,
        "split_sizes": rp_splits["split"].value_counts().to_dict(),
        "split_type": "time-based (train<=2011, validation=2012, test>=2013); no random seed",
        "training_seed": 42,  # ravenpack_sentiment.train_ravenpack default
        "split_content_sha256": {
            split_name: _hash_pairs(sub["headline"], sub["label_name"])
            for split_name, sub in rp_splits.groupby("split")
        },
    }
else:
    # PhraseBank checkpoint — trained on Financial PhraseBank (liquidAI_prep.ipynb).
    data_provenance = {
        "dataset_repo": "atrost/financial_phrasebank",
        "dataset_config": "sentences_50agree (via atrost/financial_phrasebank mirror)",
        "split_sizes": {name: raw[name].num_rows for name in raw},
        "split_type": "pre-defined splits shipped by source repo; no re-split seed",
        "training_seed": 42,  # liquidAI_prep.ipynb TrainingArguments(seed=...)
        "split_content_sha256": {
            name: _hash_pairs(raw[name]["sentence"], raw[name]["label"]) for name in raw
        },
    }
data_provenance

In [ ]:
import json
from datetime import datetime, timezone

provenance = {
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "checkpoint": {"label": ckpt_label, "path": str(model_dir.relative_to(PROJECT_ROOT))},
    "git": git_info,
    "weights": weights_snapshot,
    "model_config": config_info,
    "tokenizer": tokenizer_settings,
    "data": data_provenance,
}

provenance_path = model_dir / "provenance.json"
provenance_path.write_text(json.dumps(provenance, indent=2, default=str) + "\n", encoding="utf-8")
print(f"Saved provenance snapshot to {provenance_path.resolve()}")
print(f"  checkpoint      : {ckpt_label} ({model_dir.relative_to(PROJECT_ROOT)})")
print(f"  git commit      : {git_info['commit_hash_short']} (dirty={git_info['is_dirty']})")
print(f"  weight files    : {[w['file'] for w in weights_snapshot]}")
print(f"  num_labels      : {config_info['num_labels']}")
print(f"  max_length      : {tokenizer_settings['max_length_used']} (padding={tokenizer_settings['padding_strategy']})")
print(f"  split sizes     : {data_provenance['split_sizes']}")

## Macro-F1 comparison: in-domain PhraseBank vs out-of-domain RavenPack

Compare the same loaded checkpoint across the PhraseBank train / validation / test splits and the RavenPack evaluation split. The first chart shows overall macro-F1 and accuracy; the second chart breaks macro-F1 down by sentiment class so the domain-shift failure mode is easier to see.

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support
import plotly.express as px


required_vars = [
    "raw",
    "dataset_id2label",
    "tokenizer",
    "model",
    "device",
    "results",
    "EVAL_SPLIT",
    "ckpt_label",
]
missing = [name for name in required_vars if name not in globals()]
if missing:
    raise RuntimeError(
        "Run the earlier setup and RavenPack scoring cells first. "
        f"Missing notebook variables: {missing}"
    )


_CLASS_ORDER = ["negative", "neutral", "positive"]


def _metrics_row(dataset: str, split: str, domain: str, y_true, y_pred) -> dict:
    return {
        "dataset": dataset,
        "split": split,
        "domain": domain,
        "evaluation": f"{dataset} {split}",
        "checkpoint": ckpt_label,
        "n_rows": int(len(y_true)),
        "macro_f1": float(
            f1_score(y_true, y_pred, labels=_CLASS_ORDER, average="macro", zero_division=0)
        ),
        "accuracy": float(accuracy_score(y_true, y_pred)),
    }


def _class_f1_rows(dataset: str, split: str, domain: str, y_true, y_pred) -> list[dict]:
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=_CLASS_ORDER,
        zero_division=0,
    )
    return [
        {
            "dataset": dataset,
            "split": split,
            "domain": domain,
            "evaluation": f"{dataset} {split}",
            "label": label,
            "precision": float(p),
            "recall": float(r),
            "f1": float(f),
            "support": int(s),
        }
        for label, p, r, f, s in zip(_CLASS_ORDER, precision, recall, f1, support)
    ]


summary_rows = []
class_rows = []

for split_name in ["train", "validation", "test"]:
    split_df = raw[split_name].to_pandas()
    pred_chunks = []
    sentences = split_df["sentence"].tolist()
    for start in range(0, len(sentences), 64):
        pred_chunks.append(predict_sentences(sentences[start : start + 64], tokenizer, model, device))
    split_preds = pd.concat(pred_chunks, ignore_index=True)

    y_true = split_df["label"].map(dataset_id2label)
    y_pred = split_preds["pred"]
    summary_rows.append(_metrics_row("PhraseBank", split_name, "in-domain", y_true, y_pred))
    class_rows.extend(_class_f1_rows("PhraseBank", split_name, "in-domain", y_true, y_pred))

rp_split = EVAL_SPLIT or "all"
rp_y_true = results["actual"]
rp_y_pred = results["pred"]
summary_rows.append(_metrics_row("RavenPack", rp_split, "out-of-domain", rp_y_true, rp_y_pred))
class_rows.extend(_class_f1_rows("RavenPack", rp_split, "out-of-domain", rp_y_true, rp_y_pred))

f1_comparison = pd.DataFrame(summary_rows)
class_f1_comparison = pd.DataFrame(class_rows)

display(
    f1_comparison[["dataset", "split", "domain", "n_rows", "macro_f1", "accuracy"]]
    .style.format({"macro_f1": "{:.1%}", "accuracy": "{:.1%}", "n_rows": "{:,}"})
)

metric_long = f1_comparison.melt(
    id_vars=["dataset", "split", "domain", "evaluation", "checkpoint", "n_rows"],
    value_vars=["macro_f1", "accuracy"],
    var_name="metric",
    value_name="score",
)
metric_long["metric"] = metric_long["metric"].map({"macro_f1": "Macro-F1", "accuracy": "Accuracy"})
metric_long["score_label"] = metric_long["score"].map(lambda x: f"{x:.1%}")

fig = px.bar(
    metric_long,
    x="evaluation",
    y="score",
    color="metric",
    barmode="group",
    text="score_label",
    hover_data={"domain": True, "checkpoint": True, "n_rows": ":,", "score": ":.3f"},
    title=f"Baseline checkpoint performance: PhraseBank splits vs RavenPack ({rp_split})",
    color_discrete_map={"Macro-F1": "#2563eb", "Accuracy": "#059669"},
)
fig.update_traces(textposition="outside", cliponaxis=False)
fig.update_yaxes(title="Score", tickformat=".0%", range=[0, 1.08])
fig.update_xaxes(title="Evaluation split")
fig.update_layout(height=480, legend_title_text="Metric", margin=dict(t=70, r=30, b=70, l=60))
fig.show()

class_f1_comparison["f1_label"] = class_f1_comparison["f1"].map(lambda x: f"{x:.1%}")
fig2 = px.bar(
    class_f1_comparison,
    x="label",
    y="f1",
    color="evaluation",
    barmode="group",
    text="f1_label",
    facet_col="domain",
    hover_data={"support": ":,", "precision": ":.3f", "recall": ":.3f", "f1": ":.3f"},
    title="Class-level F1 by domain and split",
)
fig2.update_traces(textposition="outside", cliponaxis=False)
fig2.update_yaxes(title="Class F1", tickformat=".0%", range=[0, 1.08])
fig2.update_xaxes(title="Sentiment class")
fig2.update_layout(height=500, legend_title_text="Evaluation", margin=dict(t=80, r=30, b=60, l=60))
fig2.show()

precision_recall_long = class_f1_comparison.melt(
    id_vars=["dataset", "split", "domain", "evaluation", "label", "support"],
    value_vars=["precision", "recall"],
    var_name="metric",
    value_name="score",
)
precision_recall_long["metric"] = precision_recall_long["metric"].str.title()
precision_recall_long["score_label"] = precision_recall_long["score"].map(lambda x: f"{x:.1%}")

fig3 = px.bar(
    precision_recall_long,
    x="label",
    y="score",
    color="metric",
    barmode="group",
    text="score_label",
    facet_row="domain",
    facet_col="evaluation",
    hover_data={"support": ":,", "score": ":.3f"},
    title="Precision vs recall by class, domain, and split",
    color_discrete_map={"Precision": "#7c3aed", "Recall": "#ea580c"},
)
fig3.update_traces(textposition="outside", cliponaxis=False)
fig3.update_yaxes(title="Score", tickformat=".0%", range=[0, 1.08])
fig3.update_xaxes(title="Sentiment class")
fig3.update_layout(height=720, legend_title_text="Metric", margin=dict(t=90, r=30, b=60, l=60))
fig3.for_each_annotation(lambda a: a.update(text=a.text.replace("domain=", "").replace("evaluation=", "")))
fig3.show()

driver_rows = []
for row in class_f1_comparison.itertuples(index=False):
    driver = "recall" if row.recall < row.precision else "precision"
    driver_rows.append({
        "evaluation": row.evaluation,
        "domain": row.domain,
        "label": row.label,
        "precision": row.precision,
        "recall": row.recall,
        "f1": row.f1,
        "lower_component": driver,
        "precision_minus_recall": row.precision - row.recall,
    })
precision_recall_drivers = pd.DataFrame(driver_rows)
display(
    precision_recall_drivers.sort_values(["domain", "evaluation", "label"])
    .style.format({
        "precision": "{:.1%}",
        "recall": "{:.1%}",
        "f1": "{:.1%}",
        "precision_minus_recall": "{:+.1%}",
    })
)

pb_test_f1 = f1_comparison.loc[
    (f1_comparison["dataset"] == "PhraseBank") & (f1_comparison["split"] == "test"),
    "macro_f1",
].iloc[0]
rp_f1 = f1_comparison.loc[f1_comparison["dataset"] == "RavenPack", "macro_f1"].iloc[0]
print(
    "Domain-shift macro-F1 gap "
    f"(PhraseBank test → RavenPack {rp_split}): {pb_test_f1 - rp_f1:.1%} "
    f"({pb_test_f1:.1%} → {rp_f1:.1%})"
)


## Out-of-domain label prevalence: observed vs predicted

Compare the RavenPack actual label distribution against the checkpoint's predicted label distribution on the same out-of-domain evaluation rows. This makes the distribution shift visible as a prevalence gap, not just as precision / recall / F1.

In [28]:
import plotly.express as px


required_vars = ["results", "EVAL_SPLIT", "ckpt_label"]
missing = [name for name in required_vars if name not in globals()]
if missing:
    raise RuntimeError(
        "Run the RavenPack scoring cell first so `results` exists. "
        f"Missing notebook variables: {missing}"
    )

_CLASS_ORDER = ["negative", "neutral", "positive"]
rp_split = EVAL_SPLIT or "all"
n_eval = len(results)

observed_counts = results["actual"].value_counts().reindex(_CLASS_ORDER, fill_value=0)
predicted_counts = results["pred"].value_counts().reindex(_CLASS_ORDER, fill_value=0)

observed_prevalence = pd.DataFrame({
    "label": observed_counts.index.tolist(),
    "count": observed_counts.values,
    "series": "Observed / actual",
})
predicted_prevalence = pd.DataFrame({
    "label": predicted_counts.index.tolist(),
    "count": predicted_counts.values,
    "series": "Predicted by checkpoint",
})
prevalence = pd.concat([observed_prevalence, predicted_prevalence], ignore_index=True)
prevalence["pct"] = prevalence["count"] / n_eval
prevalence["pct_label"] = prevalence.apply(lambda r: f"{r['pct']:.1%}<br>n={int(r['count']):,}", axis=1)
prevalence["label"] = pd.Categorical(prevalence["label"], categories=_CLASS_ORDER, ordered=True)

display(
    prevalence.pivot(index="label", columns="series", values=["count", "pct"])
    .reindex(_CLASS_ORDER)
    .style.format({("pct", "Observed / actual"): "{:.1%}", ("pct", "Predicted by checkpoint"): "{:.1%}"})
)

fig = px.bar(
    prevalence,
    x="label",
    y="pct",
    color="series",
    barmode="group",
    text="pct_label",
    category_orders={"label": _CLASS_ORDER, "series": ["Observed / actual", "Predicted by checkpoint"]},
    hover_data={"count": ":,", "pct": ":.2%", "label": False},
    title=f"RavenPack label prevalence: observed vs predicted ({ckpt_label}, split={rp_split}, n={n_eval:,})",
    color_discrete_map={"Observed / actual": "#0f766e", "Predicted by checkpoint": "#dc2626"},
)
fig.update_traces(textposition="outside", cliponaxis=False)
fig.update_yaxes(title="Share of out-of-domain rows", tickformat=".0%", range=[0, max(0.05, prevalence["pct"].max() * 1.18)])
fig.update_xaxes(title="Sentiment label")
fig.update_layout(height=480, legend_title_text="Distribution", margin=dict(t=80, r=30, b=60, l=60))
fig.show()

gap = prevalence.pivot(index="label", columns="series", values="pct").reindex(_CLASS_ORDER)
gap["prediction_minus_actual_pp"] = (gap["Predicted by checkpoint"] - gap["Observed / actual"]) * 100
print("Prediction prevalence minus actual prevalence (percentage points):")
display(gap[["prediction_minus_actual_pp"]].style.format("{:+.1f} pp"))


KeyError: 'label'